In [ ]:
# 1. System & Python dependencies
!apt -y update -qq
!apt -y install -qq libgl1 libglib2.0-0 wget git

In [ ]:
# Clone or update C_UI6
%cd /content
!rm -rf /content/C_UI6
!git clone https://github.com/mathiu-06/C_UI6.git /content/C_UI6
%cd /content/C_UI6

In [ ]:
# Install Python packages
!pip install -r requirements.txt

### Mount Google Drive


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# import os
# import ipywidgets as widgets
# from IPython.display import display

# models_dir = '/content/drive/MyDrive/ComfyUI/models'
# model_paths = []

# # Recursively collect all model files
# if os.path.exists(models_dir):
#     for root, dirs, files in os.walk(models_dir):
#         for file_name in files:
#             # Ignore hidden checkpoints directories
#             if '.ipynb_checkpoints' not in root:
#                 rel_path = os.path.relpath(os.path.join(root, file_name), models_dir)
#                 model_paths.append(rel_path)
# model_paths.sort()

# # Initialize the global list to store the selections
# selected_models_list = []

# # Create the SelectMultiple widget
# model_selector = widgets.SelectMultiple(
#     options=model_paths,
#     description='Models:',
#     style={'description_width': 'initial'},
#     layout=widgets.Layout(width='80%', height='250px')
# )

# # Create a button to save the selection
# save_button = widgets.Button(
#     description="Save Selection",
#     button_style='success',
#     icon='check'
# )

# output_area = widgets.Output()

# # Define the button click event
# def on_button_clicked(b):
#     global selected_models_list
#     with output_area:
#         output_area.clear_output()
#         # Construct full paths and save to the list
#         selected_models_list = [os.path.join(models_dir, p) for p in model_selector.value]
#         print(f"✅ Saved {len(selected_models_list)} model(s) to the 'selected_models_list' variable:")
#         for path in selected_models_list:
#             print(f"  - {path}")

# save_button.on_click(on_button_clicked)

# # Display the widget UI
# print("Hold Ctrl/Cmd or Shift to select multiple models.")
# display(widgets.VBox([model_selector, save_button, output_area]))

In [ ]:
# import os
# import shutil
# from tqdm.notebook import tqdm
# from concurrent.futures import ThreadPoolExecutor, as_completed

# source_base_dir = '/content/drive/MyDrive/ComfyUI/models'
# target_base_dir = '/content/C_UI6/models'

# print(f"Copying {len(selected_models_list)} model(s) to {target_base_dir} using multithreading...")

# def copy_model(model_path):
#     if os.path.exists(model_path):
#         # Determine the relative path to preserve the subdirectory structure
#         rel_path = os.path.relpath(model_path, source_base_dir)
#         target_path = os.path.join(target_base_dir, rel_path)

#         # Create the target directory if it doesn't exist
#         os.makedirs(os.path.dirname(target_path), exist_ok=True)

#         shutil.copy2(model_path, target_path)
#         return True, rel_path
#     else:
#         return False, model_path

# # Using ThreadPoolExecutor for I/O bound parallel copying
# with ThreadPoolExecutor(max_workers=4) as executor:
#     # Submit all copying tasks to the thread pool
#     futures = [executor.submit(copy_model, path) for path in selected_models_list]

#     # Process completed tasks and update the progress bar
#     for future in tqdm(as_completed(futures), total=len(selected_models_list), desc="Copying models"):
#         success, path_info = future.result()
#         if not success:
#             print(f"Warning - File not found: {path_info}")

# print("✅ Copy operation complete!")

In [ ]:
# !cp -rv /content/drive/MyDrive/ComfyUI/user/default/ /content/C_UI6/user/

### xyz


In [ ]:
import subprocess
from google.colab import userdata
from pycloudflared.util import download

# 1. Get the token you saved in Colab Secrets
cf_token = userdata.get('TUNNEL_TOKEN')

try:
    # Try downloading/updating the binary
    cloudflared_bin = str(download())
except OSError as e:
    if e.errno == 26:
        print("ℹ️ cloudflared binary is busy (running in background). Reusing the existing binary.")
        # Fallback to the known installation path
        cloudflared_bin = '/usr/local/lib/python3.13/dist-packages/pycloudflared/cloudflared-linux-amd64'
    else:
        raise e

# 2. Run the Named Tunnel
print("⏳ Starting Cloudflare Named Tunnel...")
print("🔗 ComfyUI will be permanently available at: https://pc.ingeconia.xyz") # <-- Updated to match your Cloudflare setup
print("👁️ Live Read-Only Workflow Monitor: https://pc.ingeconia.xyz/live-view")

cf_proc = subprocess.Popen(
    [cloudflared_bin, 'tunnel', '--no-autoupdate', 'run', '--token', cf_token],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)



In [ ]:
# 3. Launch ComfyUI
import os
# Ensure the required custom_nodes directory exists inside ComfyUI
os.makedirs('/content/C_UI6/custom_nodes', exist_ok=True)
print("✅ Guaranteed existence of '/content/C_UI6/custom_nodes'")

# Optional: set LIVE_VIEW_TOKEN in Colab Secrets to password-protect /live-view (leave unset for open read-only access)
try:
    from google.colab import userdata
    live_view_token = userdata.get('LIVE_VIEW_TOKEN')
    if live_view_token:
        os.environ['COMFYUI_LIVE_VIEW_TOKEN'] = live_view_token
        print("🔒 Live View (/live-view) protected with LIVE_VIEW_TOKEN from Colab Secrets")
    else:
        os.environ.pop('COMFYUI_LIVE_VIEW_TOKEN', None)
        print("👁️ Live View (/live-view) open in read-only mode (no LIVE_VIEW_TOKEN set)")
except Exception:
    os.environ.pop('COMFYUI_LIVE_VIEW_TOKEN', None)
    print("👁️ Live View (/live-view) open in read-only mode (no LIVE_VIEW_TOKEN set)")

%cd /content/C_UI6
!python main.py --multi-user --listen 0.0.0.0 --port 8188 --cuda-device 0 --enable-cors-header "*"